# n006 — Artifact Phylogeny

This notebook analyses the **inheritance graph of artifacts** — who influenced whom. Two phylogeny sources are compared throughout: a hand-annotated graph (`hand`) and one produced by an LLM (`MODEL`).

**REQUIRES:** `004_artifact_analysis.py` and `006_artifact_phylogeny.py` to be run first.

**Outputs** (saved to `<ROOT>/plots/`):
- `004_artifact_depth_ccdf.pdf` — unnormalized lineage depth CCDF
- `004_artifact_distributions.pdf` — 2-panel: novelty distribution + normalized depth CCDF
- `A005_in_out_scatter.pdf` — scatter of in-degree vs out-degree colored by novelty, per run/condition
- `A005_graph_density_vs_score.pdf` — graph density as a function of LLM confidence threshold
- `A005_high_degree_scatter.pdf` — scatter of high-degree node counts (in vs out) per condition

**Phylogeny dict schema:** `{child_id: {parent_id: score, ...}}` where `score ∈ [0, 1]` is the LLM's confidence in the edge. The hand graph uses `score = 1.0` for all edges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.utils import ROOT
from core.utils.analysis_utils import get_exp_folders
from collections import defaultdict
from analysis_scripts.artifact_complexity import ExperimentArtifacts
import pandas as pd
import io
import contextlib
import json
import matplotlib.colors as mcolors
import itertools
import pickle as pkl
from tqdm import tqdm

# force reload these everytime
import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import color_map, plot_params

plt.rcParams.update(plot_params)


## Configuration

Set `EXP_BASE_NAMES` to the list of experiment base names you want to compare. All matching completed runs under `ROOT/logs/` are collected automatically.

- `PLOT_NAMES` — display-friendly labels (defaults to title-cased experiment name).
- `SHOW_STD` — toggle standard-deviation bands on time-series plots.
- `MODEL` — the phylogeny annotation model.
- `NOVELTY_MODEL` — the model used for novelty annotations.

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False
NOVELTY_MODEL = 'claude-sonnet-4-5-20250929'
MODEL = 'claude-haiku-4-5'
experiment_order = list(PLOT_NAMES.keys())


experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Data Loading

For each run, loads artifact data via `ExperimentArtifacts` then reads the LLM phylogeny and novelty scores. 

`validate_model_phylogeny()` checks for nodes whose value is not a dict or contains an `'error'` key — these indicate failed LLM calls and are reported but do not abort loading.

In [ ]:
def validate_model_phylogeny(phyl_dict):
    invalid_nodes = []
    for node, data in phyl_dict.items():
        if not isinstance(data, dict):
            invalid_nodes.append(node)
            continue
        if 'error' in data:
            invalid_nodes.append(node)
            continue
    
    if invalid_nodes:
        return False, invalid_nodes
    else:
        return True, invalid_nodes

In [ ]:
artifacts = defaultdict(list)

f = io.StringIO()

phylogeny = {}
experiment_novelties = defaultdict(list)

for base_name, exps in experiments.items():
    phylogeny[base_name] = []
    for exp in tqdm(exps, desc='Exp: ' + base_name):
        exp_path = EXP_PATH / exp
        with contextlib.redirect_stdout(f):
            art = ExperimentArtifacts(exp_path=exp_path, embedding_dimensions=512, save_path=exp_path / "artifact_analysis")
            art.load(force_recalc=False)
            art._load_raw_artifacts()

        try:
            model_phyl = json.load(open(exp_path / "artifact_analysis" / f"artifact_phylogeny_{MODEL}.json", "r"))
        except:
            print("No phylogeny found for", exp)
            continue
        valid, invalid_nodes = validate_model_phylogeny(model_phyl)
        if not valid:
            print("Invalid nodes found in phylogeny for", exp, ":", invalid_nodes)


        try:
            with open(exp_path / 'artifact_analysis' / f'novelties_{NOVELTY_MODEL}.pkl', 'rb') as nov_file:
                novelties = pkl.load(nov_file)
        except FileNotFoundError:
            print(f"Cannot load novelties from {NOVELTY_MODEL} of {exp} ")
            novelties = {}

        experiment_novelties[base_name].append(novelties)

        for art_id in art.all_artifacts:
            try:
                nov = novelties[art_id]
            except KeyError:
                nov = -1
            art.all_artifacts[art_id]['novelty'] = nov

        phylogeny[base_name].append({'hand': json.load(open(exp_path / "artifact_analysis" / "artifact_phylogeny_hand.json", "r")),
                            MODEL: model_phyl})

        artifacts[base_name].append(art)
    print()

## Graph Utilities

Core helpers for working with phylogeny graphs:

- **`to_nx_digraph(parents_by_node, artifacts, score_key)`** — converts a phylogeny dict to a `nx.DiGraph` with edges `parent → child`. Accepts both `{child: {parent: score}}` and `{child: [parent, ...]}` formats. Optionally attaches `creation_time` and `novelty` as node attributes from an `ExperimentArtifacts` dict.
- **`in_degree_metrics(G)` / `out_degree_metrics(G)`** — return (max, mean, median) of in/out degrees.
- **`graph_sources(G)`** — nodes with in-degree 0 (original artifacts with no known parents).
- **`graph_path_lengths(G)`** — longest path length in the DAG.
- **`avg_source_to_sink_distances(G)`** — average shortest path from each source to reachable sinks.
- **`filter_by_score(G, min_score)`** — returns a subgraph keeping only edges with `score ≥ min_score`; isolated nodes are preserved.
- **`style_boxplot(ax, bp, colors)`** — applies the standard translucent-box styling to a matplotlib boxplot.

In [ ]:
import networkx as nx

def to_nx_digraph(parents_by_node, artifacts=None, score_key="score"):
    """
    parents_by_node:
      - {node: {parent: score, ...}}
      - {node: [parent, parent, ...]}

    Returns:
      nx.DiGraph with edges parent -> node
      and edge attribute `score` (default = 1.0)
    """
    G = nx.DiGraph()

    # --- collect all nodes ---
    nodes = set()
    for child, parents in parents_by_node.items():
        try:
            child = int(child)
            nodes.add(child)
        except:
            continue

        if isinstance(parents, dict):
            for p in parents.keys():
                try:
                    nodes.add(int(p))
                except:
                    pass
        else:  # list / iterable
            for p in parents:
                try:
                    nodes.add(int(p))
                except:
                    pass

    G.add_nodes_from(nodes)

    # --- add node attributes ---
    if artifacts is not None:
        for n in G.nodes:
            art = artifacts.get(int(n), {})
            G.nodes[n]["creation_time"] = art.get("creation_time", None)
            G.nodes[n]["novelty"] = max(art.get("novelty", 0.0), 0.0)

    # --- add edges ---
    for child, parents in parents_by_node.items():
        try:
            child = int(child)
        except:
            continue

        if isinstance(parents, dict):
            # parent -> child with score
            for p, score in parents.items():
                try:
                    p = int(p)
                    G.add_edge(p, child, **{score_key: float(score)})
                except:
                    continue
        else:
            # parent -> child with default score = 1
            for p in parents:
                try:
                    p = int(p)
                    G.add_edge(p, child, **{score_key: 1.0})
                except:
                    continue

    return G

def in_degree_metrics(G):
    """Calculate max, mean, and median ancestors for each artifact."""
    in_deg = list(dict(G.in_degree()).values())
    return np.max(in_deg), np.mean(in_deg), np.median(in_deg)


def out_degree_metrics(G):
    """Calculate max, mean, and median children for each artifact."""
    out_deg = list(dict(G.out_degree()).values())
    return np.max(out_deg), np.mean(out_deg), np.median(out_deg)

def graph_sources(G):
    """Return list of source nodes (nodes with in-degree 0)."""
    sources =  [n for n, d in G.in_degree() if d == 0]
    return sources, len(sources)

def graph_path_lengths(G):
    return nx.dag_longest_path_length(G)

def avg_source_to_sink_distances(G):
    """
    Returns:
      per_source_avg: dict {source: average shortest path length to reachable sinks}
      overall_avg: float, average of per-source averages (unweighted)
    """
    sources = [n for n, d in G.in_degree() if d == 0]
    sinks   = {n for n, d in G.out_degree() if d == 0}

    per_source_avg = {}

    for s in sources:
        # shortest path lengths from s to all reachable nodes
        dist = nx.single_source_shortest_path_length(G, s)

        # keep only reachable sinks (exclude s itself even if it's also a sink)
        sink_dists = [d for v, d in dist.items() if v in sinks and v != s]

        if sink_dists:
            per_source_avg[s] = sum(sink_dists) / len(sink_dists)
        else:
            # no reachable sink (happens if s is isolated or only reaches itself)
            per_source_avg[s] = 0.0

    overall_avg = (sum(per_source_avg.values()) / len(per_source_avg)) if per_source_avg else 0.0
    return overall_avg

def filter_by_score(G, min_score=0.0, score_key="score"):
    """
    Return subgraph of G containing only edges with score >= min_score.
    Nodes with no remaining edges are kept as isolated nodes.
    """
    H = nx.DiGraph()
    for u, v, data in G.edges(data=True):
        score = data.get(score_key, 1.0)  # default score=1.0 if not present
        if score >= min_score:
            H.add_edge(u, v, **data)

    # Add any nodes from G that aren't in H (isolated nodes)
    for n in G.nodes:
        if n not in H:
            H.add_node(n, **G.nodes[n])

    return H

def style_boxplot(ax, bp, colors):
    # translucent boxes, no edges
    for patch, c in zip(bp["boxes"], colors):
        r, g, b = mcolors.to_rgb(c)
        patch.set_facecolor((r, g, b, 0.5))
        patch.set_edgecolor("none")

    # medians full opacity
    for med, c in zip(bp["medians"], colors):
        r, g, b = mcolors.to_rgb(c)
        med.set_color((r, g, b, 1.0))
        med.set_linewidth(2)

    # whiskers/caps (2 per box)
    rep_colors = list(itertools.chain.from_iterable([[c, c] for c in colors]))
    for whisk, c in zip(bp["whiskers"], rep_colors):
        r, g, b = mcolors.to_rgb(c)
        whisk.set_color((r, g, b, 1.0))
    for cap, c in zip(bp["caps"], rep_colors):
        r, g, b = mcolors.to_rgb(c)
        cap.set_color((r, g, b, 1.0))

    # axes formatting
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.3)

## Graphs

Converts the loaded phylogeny dicts to `nx.DiGraph` objects (LLM source only) and stores them in `graphs[base_name]`. The CCDF and degree-distribution analyses below all operate on these graph objects.

**CCDF helpers:**
- **`ccdf(values)`** — computes `P(V ≥ x)` for a list of values.
- **`node_depths_dag(G)`** — max distance from any root to each node. Cycles are broken first by removing backward edges (higher-index → lower-index) via `break_cycles_by_index()`.
- **`plot_depth_ccdf(experiments, ax, score_threshold, normalize_by_max_depth)`** — plots the depth CCDF for each condition after filtering edges below `score_threshold`.

In [ ]:
graphs = defaultdict(list)

for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G = to_nx_digraph(phyl[MODEL])
        graphs[base_name].append(G)

In [ ]:
def ccdf(values):
    """Return x,y for CCDF: P(V >= x)."""
    xs = sorted(set(values))
    n = len(values)
    ys = [sum(v >= x for v in values) / n for x in xs]
    return xs, ys

def dag_diagnostics(G):
    print("Is DAG:", nx.is_directed_acyclic_graph(G))
    try:
        cycle = nx.find_cycle(G, orientation="original")
        print("Example cycle:", cycle[:5])
    except nx.NetworkXNoCycle:
        print("No cycle found (unexpected).")

def break_cycles_by_index(G):
    """
    Iteratively find cycles and remove backward edges (u > v)
    only from edges that are part of a detected cycle.
    """
    G = G.copy()

    while not nx.is_directed_acyclic_graph(G):
        cycle = nx.find_cycle(G, orientation="original")

        # extract (u, v) pairs from cycle
        cycle_edges = [(u, v) for u, v, _ in cycle]

        # backward edges in the cycle
        backward = [(u, v) for u, v in cycle_edges if u > v]

        if not backward:
            raise RuntimeError(
                "Cycle found but no backward edge by index; "
                "node IDs may not reflect creation order."
            )

        # remove one backward edge (or all, see note below)
        G.remove_edge(*backward[0])

    return G

def node_depths_dag(G):
    """
    Depth(node) = max distance from any root to node.
    Roots = nodes with in-degree 0.
    Requires DAG.
    """
    while not nx.is_directed_acyclic_graph(G):
        dag_diagnostics(G)
        G = break_cycles_by_index(G)

    roots = [n for n in G.nodes if G.in_degree(n) == 0]
    # init: roots at depth 0; others -inf
    depth = {n: (0 if n in roots else float("-inf")) for n in G.nodes}

    # DP over topological order
    for u in nx.topological_sort(G):
        du = depth[u]
        if du == float("-inf"):
            # unreachable from any root (shouldn't happen if roots defined well)
            continue
        for v in G.successors(u):
            depth[v] = max(depth[v], du + 1)

    # If you can have multiple components, some nodes might still be -inf if no roots were found.
    # Fallback: set them to 0.
    depths = [0 if d == float("-inf") else int(d) for d in depth.values()]
    return depths

def plot_depth_ccdf(experiments, ax, score_threshold=0.0, normalize_by_max_depth=True, show_max_depth_in_legend=False):
    for cond, graphs in experiments.items():
        all_depths = []
        for G in tqdm(graphs, desc=f'{cond} '):
            filtered_G = filter_by_score(G, min_score=score_threshold)
            all_depths.extend(node_depths_dag(filtered_G))
        max_depths = np.max(all_depths)
        if normalize_by_max_depth:
            x, y = ccdf(np.array(all_depths) / max_depths)  # normalize by max depth for comparability
        else:
            x, y = ccdf(all_depths)
        
        # Add max depth to label if requested
        label = PLOT_NAMES[cond]
        if show_max_depth_in_legend:
            label = f"{label} (max={int(max_depths)})"
        
        ax.plot(x, y, label=label, color=color_map[cond])
        print(f"{cond}: max depth = {max_depths}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_depth_ccdf(graphs, ax, score_threshold=0.7, normalize_by_max_depth=False)
# ax.set_title("Artifact Depth CCDF")
ax.set_xlabel("Lineage Depth")
ax.set_ylabel("Fraction of artifacts with depth ≥ x")
ax.set_title("Unnormalized artifact Lineage Depth Distribution", loc='left')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "plots" / "004_artifact_depth_ccdf.pdf", dpi=300)
plt.show()

## Novelty Distribution + Depth CCDF (Combined Figure)

Two-panel figure combining the two key distributional views:
- **(a) Novelty distribution** — fraction of artifacts in each novelty range on a log scale (same binning as n004, `novelty_range_handling = '(]'`).
- **(b) Lineage depth CCDF** — normalized by each condition's own max depth, with max depth shown in the legend. Edges below `score_threshold = 0.7` are excluded.

Saved as `004_artifact_distributions.pdf`.

In [ ]:
novelty_ranges = [(0, 0), (0, 3), (3, 4.2), (4.2, float('inf'))]  # Define novelty ranges

novelty_range_handling = '(]' # Define whether ranges are '()' or '[]' or '(]' or '[)'

rows = []

for base_name, experiment_data in artifacts.items():
    for run_id, exp in enumerate(experiment_data):

        for r in novelty_ranges:
            lower, upper = r
            count = 0

            for art in exp.all_artifacts.values():
                nov = max(art.get("novelty", 0), 0)
                if lower != upper:
                    if novelty_range_handling == '[)':
                        if lower <= nov < upper:
                            count += 1
                    elif novelty_range_handling == '(]':
                        if lower < nov <= upper:
                            count += 1
                else:
                    if nov == lower:
                        count += 1
                

            normalize_by_length = count / exp.last_ts
            normalize_by_total = count / len(exp.all_artifacts)

            rows.append({
                "experiment": base_name,
                "run": run_id,              # seed index
                "novelty_lower": lower,
                "novelty_upper": upper,
                "novelty_range": r,         # tuple (lower, upper)
                "count": count,
                "norm_length": normalize_by_length,
                "norm_total": normalize_by_total
            })

df = pd.DataFrame(rows)

df_mean = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(mean_count=("count", "mean"), mean_norm_length=("norm_length", "mean"), mean_norm_total=("norm_total", "mean"))
)

df_std = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(std_count=("count", "std"), std_norm_length=("norm_length", "std"), std_norm_total=("norm_total", "std"))
)

import scipy.stats as stats

df_ci = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(
        se_count=("count", lambda x: stats.sem(x)),
        se_norm_length=("norm_length", lambda x: stats.sem(x)),
        se_norm_total=("norm_total", lambda x: stats.sem(x))
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for exp_name in experiment_novelties.keys():
    g = df_mean.groupby("experiment").get_group(exp_name).sort_values(
        by="novelty_range",
        key=lambda x: x.apply(lambda r: r[0])
    )

    axes[0].plot(
        g["novelty_range"].astype(str),
        g["mean_norm_total"],
        marker="o",
        label=PLOT_NAMES[exp_name],
        color=color_map[exp_name],
        linestyle='-.',
        linewidth=0.5,
    )

    if SHOW_STD:
        axes[0].fill_between(
            g["novelty_range"].astype(str),
            g["mean_norm_total"] - df_std.loc[
                (df_std["experiment"] == exp_name),
                "std_norm_total"
            ].values,
            g["mean_norm_total"] + df_std.loc[
                (df_std["experiment"] == exp_name),
                "std_norm_total"
            ].values,
            color=color_map[exp_name],
            alpha=0.2
        )

    se_values = df_ci.loc[
        (df_ci["experiment"] == exp_name),
        "se_norm_total"
    ].values

    yerr_lower = se_values  # distance below mean
    yerr_upper = se_values  # distance above mean

if novelty_range_handling == '(]':
    xlabels = [f"({x}, {y}]" if y != float('inf') else f"> {x}" for x, y in novelty_ranges]
elif novelty_range_handling == '[)':
    xlabels = [f"[{x}, {y})" if y != float('inf') else f"=> {x}" for x, y in novelty_ranges]

xlabels[0] = f"= 0"

plot_depth_ccdf(graphs, axes[1], score_threshold=0.7, normalize_by_max_depth=True, show_max_depth_in_legend=True)


axes[0].set_xlabel("Novelty range")
axes[0].set_ylabel("Fraction of artifacts (log scale)")
# axes[0].set_title("Normalized artifact count by novelty range")
axes[0].set_xticklabels(xlabels)
axes[0].legend()
axes[0].set_yscale("log")

axes[1].set_xlabel("Normalized Depth")
axes[1].set_ylabel("Fraction of artifacts with depth ≥ x")
# axes[1].legend(loc='best', fontsize=8, framealpha=0.9)

axes[0].set_title(r"$\bf{a}$ Artifact Novelty Distribution", loc='left')
axes[1].set_title(r"$\bf{b}$ Artifact Lineage Depth Distribution", loc='left')


plt.tight_layout()
plt.subplots_adjust(wspace=0.3)  # Add horizontal space between subplots
plt.savefig(ROOT / "plots" / "004_artifact_distributions.pdf", dpi=300)
plt.show()

## Phylogeny Metrics — Boxplots

Three sets of 2-row × 3-column boxplot grids comparing the hand and LLM phylogenies side by side. `SHOW_HAND` toggles the hand row.

1. **In-degree (ancestors)** — max / mean / median number of parents per artifact.
2. **Out-degree (children)** — max / mean / median number of derived artifacts.
3. **Path lengths** — number of source nodes (in-degree 0), average source-to-sink distance, and longest path in the DAG.

**`plot_scatter(ax, metrics_df, xcol, ycol, ...)`** — general scatter helper used to compare hand vs model metrics. Plots per-run points (faint), per-condition means with IQR whiskers, and a labelled legend. Supports `"iqr"`, `"std"`, `"sem"`, or `"ci95"` whisker styles.

In [ ]:
SHOW_HAND = True
if SHOW_HAND:
    rows = 2
else:
    rows = 1

fig, axs = plt.subplots(rows, 3, figsize=(4 * 4.5, 5), sharey=False)
if rows == 1:
    axs = axs[np.newaxis, :]

rows = []
for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G_hand = to_nx_digraph(phyl['hand'])
        max_hand, mean_hand, median_hand = in_degree_metrics(G_hand)
        
        # Model metrics
        G_model = to_nx_digraph(phyl[MODEL])
        max_model, mean_model, median_model = in_degree_metrics(G_model)
        
        rows.append({
            'base_name': base_name,
            'idx': idx,
            'max_hand': max_hand,
            'mean_hand': mean_hand,
            'median_hand': median_hand,
            'max_model': max_model,
            'mean_model': mean_model,
            'median_model': median_model
        })

# Create DataFrame
metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.set_index('base_name')[['max_hand', 'mean_hand', 'median_hand', 
                                                       'max_model', 'mean_model', 'median_model']]

stat_cols = ["max", "mean", "median"]
row_specs = [("model", 0), ("hand", 1)]
order = experiment_order  # or experiment_order + ["random"]

for j, s in enumerate(stat_cols):
    for suffix, i in row_specs:
        ax = axs[i, j]
        col = f"{s}_{suffix}"

        df = metrics_df[metrics_df.index.isin(order)]
        df.index = pd.Categorical(df.index, categories=order, ordered=True)

        exps_present = [bn for bn in order if bn in df.index.unique()]
        data_groups = [df.loc[bn, col] if bn in df.index else np.array([]) for bn in exps_present]
        positions = np.arange(len(exps_present))
        colors = [color_map.get(bn, (50/255., 50/255., 50/255.)) for bn in exps_present]

        bp = ax.boxplot(
            data_groups,
            positions=positions,
            patch_artist=True,
            showfliers=False,
            showcaps=False,
            whis=1,
            medianprops={"linewidth": 2},
            whiskerprops={"linewidth": 1},
            capprops={"linewidth": 1},
            widths=0.8
        )

        # mean markers
        for x, vals in zip(positions, data_groups):
            ax.plot(
                x, np.mean(vals),
                marker="o", markersize=6,
                markerfacecolor="white",
                markeredgecolor="black",
                markeredgewidth=1.3,
                zorder=5
            )

        style_boxplot(ax, bp, colors)

        ax.set_title(f"{s} ({suffix})")
        ax.set_xticks(positions)
        ax.set_xticklabels([PLOT_NAMES.get(bn, bn) for bn in exps_present],
                           rotation=45, ha="right")

plt.suptitle("Ancestor (in-degree) Metrics for Artifact Phylogenies", fontsize=16)
plt.tight_layout()

In [ ]:
SHOW_HAND = True
if SHOW_HAND:
    rows = 2
else:
    rows = 1

fig, axs = plt.subplots(rows, 3, figsize=(4 * 4.5, 5), sharey=False)
if rows == 1:
    axs = axs[np.newaxis, :]

rows = []
for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G_hand = to_nx_digraph(phyl['hand'])
        max_hand, mean_hand, median_hand = out_degree_metrics(G_hand)
        
        # Model metrics
        G_model = to_nx_digraph(phyl[MODEL])
        max_model, mean_model, median_model = out_degree_metrics(G_model)
        
        rows.append({
            'base_name': base_name,
            'idx': idx,
            'max_hand': max_hand,
            'mean_hand': mean_hand,
            'median_hand': median_hand,
            'max_model': max_model,
            'mean_model': mean_model,
            'median_model': median_model
        })

# Create DataFrame
metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.set_index('base_name')[['max_hand', 'mean_hand', 'median_hand', 
                                                       'max_model', 'mean_model', 'median_model']]

stat_cols = ["max", "mean", "median"]
row_specs = [("model", 0), ("hand", 1)]
order = experiment_order  # or experiment_order + ["random"]

for j, s in enumerate(stat_cols):
    for suffix, i in row_specs:
        ax = axs[i, j]
        col = f"{s}_{suffix}"

        df = metrics_df[metrics_df.index.isin(order)]
        df.index = pd.Categorical(df.index, categories=order, ordered=True)

        exps_present = [bn for bn in order if bn in df.index.unique()]
        data_groups = [df.loc[bn, col] if bn in df.index else np.array([]) for bn in exps_present]
        positions = np.arange(len(exps_present))
        colors = [color_map.get(bn, (50/255., 50/255., 50/255.)) for bn in exps_present]

        bp = ax.boxplot(
            data_groups,
            positions=positions,
            patch_artist=True,
            showfliers=False,
            showcaps=False,
            whis=1,
            medianprops={"linewidth": 2},
            whiskerprops={"linewidth": 1},
            capprops={"linewidth": 1},
            widths=0.8
        )

        # mean markers
        for x, vals in zip(positions, data_groups):
            ax.plot(
                x, np.mean(vals),
                marker="o", markersize=6,
                markerfacecolor="white",
                markeredgecolor="black",
                markeredgewidth=1.3,
                zorder=5
            )

        style_boxplot(ax, bp, colors)

        ax.set_title(f"{s} ({suffix})")
        ax.set_xticks(positions)
        ax.set_xticklabels([PLOT_NAMES.get(bn, bn) for bn in exps_present],
                           rotation=45, ha="right")

plt.suptitle("Children (out-degree) Metrics for Artifact Phylogenies", fontsize=16)
plt.tight_layout()

In [ ]:
SHOW_HAND = True
if SHOW_HAND:
    rows = 2
else:
    rows = 1

fig, axs = plt.subplots(rows, 3, figsize=(4 * 4.5, 5), sharey=False)
if rows == 1:
    axs = axs[np.newaxis, :]


rows = []
for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G_hand = to_nx_digraph(phyl['hand'])
        _, num_sources_hand = graph_sources(G_hand)
        max_len_hand = graph_path_lengths(G_hand)
        avg_len_hand = avg_source_to_sink_distances(G_hand)
        
        # Model metrics
        G_model = to_nx_digraph(phyl[MODEL])
        _, num_sources_model = graph_sources(G_model)
        try:
            max_len_model = graph_path_lengths(G_model)
        except:
            print("IDX:", idx, "BASE NAME:", base_name)
            max_len_model = np.nan
        avg_len_model = avg_source_to_sink_distances(G_model)
        
        rows.append({
            'base_name': base_name,
            'num_sources_hand': num_sources_hand,
            'avg_len_hand': avg_len_hand,
            'max_len_hand': max_len_hand,
            'num_sources_model': num_sources_model,
            'avg_len_model': avg_len_model,
            'max_len_model': max_len_model,
        })

# Create DataFrame
metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.set_index('base_name')[['num_sources_hand', 'avg_len_hand', 'max_len_hand', 
                                                       'num_sources_model', 'avg_len_model', 'max_len_model']]

stat_cols = ["num_sources", "avg_len", "max_len"]
row_specs = [("model", 0), ("hand", 1)]
order = experiment_order  # or experiment_order + ["random"]

for j, s in enumerate(stat_cols):
    for suffix, i in row_specs:
        ax = axs[i, j]
        col = f"{s}_{suffix}"

        df = metrics_df[metrics_df.index.isin(order)]
        df.index = pd.Categorical(df.index, categories=order, ordered=True)

        exps_present = [bn for bn in order if bn in df.index.unique()]
        data_groups = [df.loc[bn, col] if bn in df.index else np.array([]) for bn in exps_present]
        positions = np.arange(len(exps_present))
        colors = [color_map.get(bn, (50/255., 50/255., 50/255.)) for bn in exps_present]

        bp = ax.boxplot(
            data_groups,
            positions=positions,
            patch_artist=True,
            showfliers=False,
            showcaps=False,
            whis=1,
            medianprops={"linewidth": 2},
            whiskerprops={"linewidth": 1},
            capprops={"linewidth": 1},
            widths=0.8
        )

        # mean markers
        for x, vals in zip(positions, data_groups):
            ax.plot(
                x, np.nanmean(vals),
                marker="o", markersize=6,
                markerfacecolor="white",
                markeredgecolor="black",
                markeredgewidth=1.3,
                zorder=5
            )

        style_boxplot(ax, bp, colors)

        ax.set_title(f"{s} ({suffix})")
        ax.set_xticks(positions)
        ax.set_xticklabels([PLOT_NAMES.get(bn, bn) for bn in exps_present],
                           rotation=45, ha="right")

plt.suptitle("Path Length Metrics", fontsize=16)
plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.colors as mcolors

def plot_scatter(
    ax,
    metrics_df,
    xcol, ycol,
    xlab="Mean out-degree", ylab="Mean in-degree",
    order=None,                 # list of base_names in desired legend order
    palette=None,               # list of colors aligned with `order`
    labels=None,                # list of base_names aligned with `order` (or None -> use order)
    show_runs=True,
    whiskers="iqr",             # "iqr", "std", "sem", or "ci95"
    whisker_alpha=0.5,
    whisker_lw=1.0,
    whisker_capsize=3,
    run_s=25,
    run_alpha=0.25,
    mean_s=110,
):
    """
    Scatter: x=out, y=in.
    - faint points for each run (per base_name)
    - big point for mean per base_name
    - optional whiskers (default: IQR like your plot_pareto)
    """

    # Ensure we can group by base_name even if it's the index
    df = metrics_df.copy()
    if df.index.name != "base_name":
        df.index.name = "base_name"

    # Establish order/labels
    if order is None:
        order = list(dict.fromkeys(df.index.tolist()))  # stable unique
    if labels is None:
        labels = order

    # Precompute stats (mean/std/count) and IQR (q1/q3) per base_name
    stats = df.groupby(level=0)[[xcol, ycol]].agg(["mean", "std", "count"])

    q_df = (
        df.groupby(level=0)[[xcol, ycol]]
          .quantile([0.25, 0.75])
          .unstack(level=1)  # columns like (xcol, 0.25), (xcol, 0.75), ...
    )

    # 1) Per-run cloud (faint)
    if show_runs:
        for exp in order:
            sub = df.loc[exp]
            # if only one row, .loc gives Series; make it 1-row frame
            if hasattr(sub, "to_frame") and not hasattr(sub, "columns"):
                sub = sub.to_frame().T

            ax.scatter(
                sub[xcol], sub[ycol],
                s=run_s,
                alpha=run_alpha,
                facecolors=mcolors.to_rgba(palette[exp], 0.35),
                edgecolors="none",
                zorder=1,
            )

    # 2) Means + whiskers
    for exp, lab in zip(order, labels):
        if exp not in stats.index:
            continue

        mx = float(stats.loc[exp, (xcol, "mean")])
        my = float(stats.loc[exp, (ycol, "mean")])

        sx = float(stats.loc[exp, (xcol, "std")]) if not np.isnan(stats.loc[exp, (xcol, "std")]) else 0.0
        sy = float(stats.loc[exp, (ycol, "std")]) if not np.isnan(stats.loc[exp, (ycol, "std")]) else 0.0
        n  = int(stats.loc[exp, (xcol, "count")])

        if whiskers == "std":
            ex, ey = sx, sy

        elif whiskers == "sem":
            ex = sx / np.sqrt(n) if n > 0 else 0.0
            ey = sy / np.sqrt(n) if n > 0 else 0.0

        elif whiskers == "ci95":
            # small-n friendly; match your shortcut
            t = 2.776 if n == 5 else 2.0
            ex = t * (sx / np.sqrt(n)) if n > 1 else 0.0
            ey = t * (sy / np.sqrt(n)) if n > 1 else 0.0

        elif whiskers == "iqr" and exp in q_df.index:
            x_q1 = float(q_df.loc[exp, (xcol, 0.25)])
            x_q3 = float(q_df.loc[exp, (xcol, 0.75)])
            y_q1 = float(q_df.loc[exp, (ycol, 0.25)])
            y_q3 = float(q_df.loc[exp, (ycol, 0.75)])

            left  = max(0.0, mx - x_q1)
            right = max(0.0, x_q3 - mx)
            down  = max(0.0, my - y_q1)
            up    = max(0.0, y_q3 - my)
            ex = [[left], [right]]
            ey = [[down], [up]]

        else:
            raise ValueError('whiskers must be one of: "iqr", "std", "sem", "ci95"')

        # whiskers
        ax.errorbar(
            mx, my,
            xerr=ex, yerr=ey,
            fmt="none",
            ecolor=mcolors.to_rgba(palette[exp], whisker_alpha),
            elinewidth=whisker_lw,
            capsize=whisker_capsize,
            capthick=whisker_lw,
            zorder=2,
        )

        # mean marker (big, black edge)
        ax.scatter(
            mx, my,
            s=mean_s,
            alpha=1.0,
            facecolors=mcolors.to_rgba(palette[exp], 1.0),
            edgecolors="black",
            linewidths=1.2,
            zorder=3,
            label=lab,
        )

    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3)
    return stats

In [ ]:
SHOW_HAND = True
if SHOW_HAND:
    rows = 2
else:
    rows = 1

fig, axs = plt.subplots(1, 1, figsize=(5, 5), sharey=False)

rows = []
for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G_hand = to_nx_digraph(phyl['hand'])
        max_out_hand, mean_out_hand, median_out_hand = out_degree_metrics(G_hand)
        max_in_hand, mean_in_hand, median_in_hand = in_degree_metrics(G_hand)
        
        # Model metrics
        G_model = to_nx_digraph(phyl[MODEL])
        max_out_model, mean_out_model, median_out_model = out_degree_metrics(G_model)
        max_in_model, mean_in_model, median_in_model = in_degree_metrics(G_model)
        
        rows.append({
            'base_name': base_name,
            'idx': idx,
            'max_out_hand': max_out_hand,
            'mean_out_hand': mean_out_hand,
            'median_out_hand': median_out_hand,
            'max_out_model': max_out_model,
            'mean_out_model': mean_out_model,
            'median_out_model': median_out_model,
            'max_in_hand': max_in_hand,
            'mean_in_hand': mean_in_hand,
            'median_in_hand': median_in_hand,
            'max_in_model': max_in_model,
            'mean_in_model': mean_in_model,
            'median_in_model': median_in_model
        })

# Create DataFrame
metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.set_index('base_name')[['mean_in_hand', 'mean_out_hand', 
                                                       'mean_out_model', 'mean_in_model']]

row_specs = [("model", 0), ("hand", 1)]
order = experiment_order  # or experiment_order + ["random"]

plot_scatter(
    axs,
    metrics_df,
    xcol="mean_out_model",
    ycol="mean_out_hand",
    xlab="Mean out-degree (model)",
    ylab="Mean out-degree (hand)",
    order=experiment_order,     # your desired order
    palette=color_map,            # same palette you already use
    labels=[PLOT_NAMES.get(e, e) for e in experiment_order],
    show_runs=True,
    whiskers="iqr",
)

axs.set_title("Hand-Model Artifact Phylogeny")

handles, leg_labels = axs.get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="lower center", ncol=4, frameon=True,
           bbox_to_anchor=(0.5, -0.1)  # push legend below the figure
)

plt.tight_layout()
plt.show()

## Connections

Explores **degree structure and edge confidence** in the LLM phylogeny graphs.

- **In/out-degree scatter** (`A005_in_out_scatter.pdf`) — each node plotted at (out-degree, in-degree), coloured by novelty score using a viridis colormap. One subplot per run/condition; high-novelty artifacts tend to appear as hubs.
- **Graph density vs LLM confidence threshold** (`A005_graph_density_vs_score.pdf`) — shows how many edges survive as the `score ≥ threshold` cut-off is raised. Useful for choosing a sensible threshold for downstream analyses.
- **Out-degree ridge plot** — per-seed normalised out-degree distributions stacked as a ridge plot. Highlights differences in the shape of the degree distribution across conditions.
- **High-degree scatter** (`A005_high_degree_scatter.pdf`) — for a fixed `score_threshold` and `DEGREE_THRESHOLD`, counts nodes with out-degree > threshold (x) vs in-degree > threshold (y) per run, then plots per-condition means with IQR whiskers.

In [ ]:
def scatter_in_out_degree(G, ax, color_key='novelty', sort=True):
    """
    Scatter nodes where:
      x = out-degree
      y = in-degree
    """
    x = []
    y = []
    cs = []

    data = []
    for n in G.nodes:
        value = G.nodes[n].get(color_key, 0.0)
        data.append((
            value,
            G.out_degree(n),
            G.in_degree(n),
        ))

    # Sort by novelty (low → high)
    if sort:
        data.sort(key=lambda x: x[0])

    values, xs, ys = zip(*data)

    ax.scatter(xs, ys, c=values, cmap='viridis', vmin=0.0, vmax=max(values))
    ax.set_xlabel("Out-degree")
    ax.set_ylabel("In-degree")
    ax.set_ylim(-10, 200)
    ax.set_xlim(-10, 200)
    ax.grid(True)

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

fig, axs = plt.subplots(5, len(EXP_BASE_NAMES), figsize=(3*len(EXP_BASE_NAMES), 3*5))

for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        # Hand metrics
        G_hand = to_nx_digraph(phyl[MODEL], artifacts=artifacts[base_name][idx].all_artifacts)

        scatter_in_out_degree(G_hand, axs[idx, EXP_BASE_NAMES.index(base_name)], color_key='novelty', sort=True)

    axs[0, EXP_BASE_NAMES.index(base_name)].set_title(f"{PLOT_NAMES.get(base_name, base_name)}")

plt.tight_layout()

# Add colorbar
norm = Normalize(vmin=0, vmax=5)
sm = ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs, label='Novelty', fraction=0.046, pad=0.04, aspect=40)
plt.savefig(ROOT / "plots" / "A005_in_out_scatter.pdf", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
def graph_density_vs_score(
    G,
    thresholds,
    score_key="score",
    use_all_nodes=True,
):
    """
    Plots x = score threshold in [0,1]
          y = density of the graph after keeping edges with score >= threshold

    Params
    ------
    G : nx.DiGraph or nx.Graph
    score_key : edge attribute name storing the score (0..1)
    num_points : how many thresholds to evaluate
    use_all_nodes : if True, density computed using all nodes in G
                    if False, density computed on the induced subgraph of nodes
                    that remain after filtering edges (nodes with degree>0).
    """
    densities = []

    # Pre-cache edges and their scores for speed
    edges_scores = []
    for u, v, data in G.edges(data=True):
        s = data.get(score_key, 1.0)  # default to 1.0 if missing
        edges_scores.append((u, v, float(s)))

    all_nodes = list(G.nodes)

    for t in thresholds:
        # Build filtered graph at threshold t
        H = G.__class__()  # keeps DiGraph vs Graph type

        if use_all_nodes:
            H.add_nodes_from(all_nodes)

        # Keep edges with score >= t
        kept_edges = [(u, v, {score_key: s}) for (u, v, s) in edges_scores if s >= t]
        H.add_edges_from(kept_edges)

        if not use_all_nodes:
            # Drop isolated nodes (optional)
            active_nodes = [n for n in H.nodes if H.degree(n) > 0]
            H = H.subgraph(active_nodes).copy()

        # Density (NetworkX handles directed/undirected correctly)
        dens = nx.density(H) if H.number_of_nodes() > 1 else 0.0
        densities.append(dens)

    return np.array(densities)

In [ ]:
thresholds = np.linspace(0, 1, 100)

fig, axs = plt.subplots(1, 1, figsize=(6, 6))


for j, base_name in enumerate(EXP_BASE_NAMES):
    densities = []
    # Build graphs for all seeds in this experiment
    for idx, phyl in enumerate(phylogeny[base_name]):
        G = to_nx_digraph(phyl[MODEL], artifacts=artifacts[base_name][idx].all_artifacts)

        densities.append(graph_density_vs_score(
            G,
            thresholds,
            score_key="score",
            use_all_nodes=True,
        ))

    mean_densities = np.mean(densities, axis=0)
    axs.plot(thresholds, mean_densities, label=PLOT_NAMES[base_name],
             color=color_map[base_name])
    
axs.set_xlabel("LLM confidence Threshold")
axs.set_ylabel("Graph Density")
axs.grid(True)
axs.legend()
plt.tight_layout()
plt.savefig(ROOT / "plots" / "A005_graph_density_vs_score.pdf", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

fig, axs = plt.subplots(1, 1, figsize=(7, 5), sharey=False)

threshold = 0.

# per_base_name -> list of Counters (one per seed)
per_base_hists = defaultdict(list)

for base_name in EXP_BASE_NAMES:
    for idx, phyl in enumerate(phylogeny[base_name]):
        G_model = to_nx_digraph(phyl[MODEL])

        edges_scores = []
        for u, v, data in G_model.edges(data=True):
            s = data.get("score", 1.0)
            edges_scores.append((u, v, float(s)))

        H = G_model.__class__()
        kept_edges = [(u, v, {"score": s}) for (u, v, s) in edges_scores if s >= threshold]
        H.add_edges_from(kept_edges)

        out_degrees = [d for _, d in H.out_degree()]
        hist = Counter(out_degrees)

        per_base_hists[base_name].append(hist)

# ---- common x-axis (union of all degree values) ----

all_degree_vals = set()
for hlist in per_base_hists.values():
    for h in hlist:
        all_degree_vals.update(h.keys())

x = np.array(sorted(all_degree_vals))

def counter_to_prob_array(c: Counter, xvals: np.ndarray) -> np.ndarray:
    total = sum(c.values())
    if total == 0:
        return np.zeros_like(xvals, dtype=float)
    return np.array([c.get(int(k), 0) for k in xvals], dtype=float) / total

def normalize(v):
    m = v.max()
    return v / m if m > 0 else v

# ---- plotting ----

ridge_gap = 1.5
ridge_height = 0.9

base_order = list(EXP_BASE_NAMES)

for i, base_name in enumerate(base_order):
    y0 = i * ridge_gap

    for seed_hist in per_base_hists[base_name]:
        y = normalize(counter_to_prob_array(seed_hist, x)) * ridge_height

        # line per seed
        axs.plot(
            x,
            y0 + y,
            alpha=0.35,
            linewidth=1.2,
        )

    # optional: faint baseline
    axs.hlines(y0, x.min(), x.max(), linewidth=0.5, alpha=0.3)

# ---- cosmetics ----

axs.set_yticks([i * ridge_gap for i in range(len(base_order))])
axs.set_yticklabels(base_order)
axs.set_xlabel("Out-degree")
axs.set_ylabel("Experiment (base_name)")
axs.set_title(f"Out-degree distributions per seed (score ≥ {threshold})")

axs.spines["top"].set_visible(False)
axs.spines["right"].set_visible(False)
axs.margins(x=0.02, y=0.05)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

threshold = .7
DEGREE_THRESHOLD = 30

degree_data_dict = defaultdict(list)  # base_name -> list of counts (one per seed)

# per_base_name -> list of counts (one per seed)
per_base_counts = defaultdict(list)

for base_name in EXP_BASE_NAMES:
    for i, phyl in enumerate(phylogeny[base_name]):
        G_model = to_nx_digraph(phyl[MODEL])

        edges_scores = []
        for u, v, data in G_model.edges(data=True):
            s = data.get("score", 1.0)
            edges_scores.append((u, v, float(s)))

        H = G_model.__class__()
        kept_edges = [(u, v, {"score": s}) for (u, v, s) in edges_scores if s >= threshold]
        H.add_edges_from(kept_edges)

        out_degrees = [d for _, d in H.out_degree()]
        out_degree = sum(d > DEGREE_THRESHOLD for d in out_degrees)
        in_degrees = [d for _, d in H.in_degree()]
        in_degree = sum(d > DEGREE_THRESHOLD for d in in_degrees)

        degree_data_dict['base_name'].append(base_name)
        degree_data_dict['out_degree'].append(out_degree)
        degree_data_dict['in_degree'].append(in_degree)
        degree_data_dict['idx'].append(i)

data_df = pd.DataFrame(degree_data_dict)
data_df.set_index('base_name', inplace=True)

plot_scatter(
    ax,
    data_df,
    xcol="out_degree",
    ycol="in_degree",
    xlab="Mean out-degree",
    ylab="Mean in-degree",
    palette=color_map,            # same palette you already use
    labels=PLOT_NAMES.values(),  # use display names in legend
    show_runs=True,
    whiskers="iqr",
)

# ax.set_title(f"High-degree nodes (score ≥ {threshold} - minimum degree > {DEGREE_THRESHOLD})")
ax.legend(loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.27))

# ax.set_xlim(-5, 80)
# ax.set_ylim(-5, 90)
plt.tight_layout()
plt.savefig(ROOT / "plots" / "A005_high_degree_scatter.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Trees — Time-Aligned Graph Visualisation

**`draw_time_aligned_graph(G, ax, score_threshold, ...)`** lays out the artifact DAG with creation time on the x-axis. Nodes at the same timestep are spread evenly on the y-axis (with optional jitter). Edges are drawn as a `LineCollection` and optionally coloured by confidence score.

Key parameters:
- `score_threshold` — only edges with `score ≥` this value are drawn.
- `color_edges_by_score` — colour edges by confidence using the viridis colormap.
- `max_edges` — cap on total edges drawn (keeps densely-connected graphs readable).
- `rasterize_edges` — rasterises the edge layer to keep PDF file sizes manageable.

The grid below shows all runs for all conditions (rows = seeds, columns = conditions).

In [ ]:
import matplotlib as mpl
from matplotlib.collections import LineCollection

def draw_time_aligned_graph(
    G,
    ax,
    score_threshold=0.0,
    score_key="score",
    time_key="creation_time",
    *,
    node_size=18,
    node_alpha=0.90,
    edge_alpha=0.3,              # if None, auto-scale based on edge count
    edge_lw=0.6,
    jitter_y=0.0,
    seed=0,                       # reproducible jitter
    show_time_ticks=False,
    color_edges_by_score=True,
    cmap="viridis",
    score_clip_quantiles=(0.02, 0.98),  # robust color scaling
    max_edges=None,               # optionally cap edges (e.g., 20000) to avoid hairballs
    rasterize_edges=True,         # keeps PDF small when tons of edges
    title=None,
    color_bar=False,
    node_color="blue",
):
    # ---------------------------
    # 1) filter edges by threshold
    # ---------------------------
    kept = []
    for u, v, data in G.edges(data=True):
        s = float(data.get(score_key, 1.0))
        if s >= score_threshold:
            kept.append((u, v, data, s))

    if max_edges is not None and len(kept) > max_edges:
        # keep strongest edges only (usually what you want visually)
        kept.sort(key=lambda x: x[3], reverse=True)
        kept = kept[:max_edges]

    # Build filtered graph (same type)
    H = G.__class__()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from([(u, v, d) for (u, v, d, s) in kept])

    # ---------------------------
    # 2) build positions
    # ---------------------------
    def get_t(n):
        t = G.nodes[n].get(time_key, None)
        return float("inf") if t is None else float(t)

    nodes_sorted = sorted(
        H.nodes,
        key=lambda n: (get_t(n), int(n) if str(n).isdigit() else str(n))
    )

    buckets = {}
    for n in nodes_sorted:
        buckets.setdefault(get_t(n), []).append(n)

    rng = np.random.default_rng(seed)

    pos = {}
    for t, ns in buckets.items():
        k = len(ns)
        ys = np.linspace(-(k - 1) / 2.0, (k - 1) / 2.0, k) if k > 1 else np.array([0.0])
        if jitter_y and k > 1:
            ys = ys + rng.uniform(-jitter_y, jitter_y, size=k)
        for y, n in zip(ys, ns):
            pos[n] = (t, float(y))

    # ---------------------------
    # 3) draw edges nicely (LineCollection)
    # ---------------------------
    if edge_alpha is None:
        # auto edge alpha: fewer edges => more visible; many edges => faint
        m = max(1, len(kept))
        edge_alpha = float(np.clip(30.0 / np.sqrt(m), 0.02, 0.20))

    segments = []
    scores = []
    for u, v, data, s in kept:
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        segments.append([(x1, y1), (x2, y2)])
        scores.append(s)

    if len(segments) > 0:
        if color_edges_by_score:
            scores = np.asarray(scores, dtype=float)
            loq, hiq = score_clip_quantiles
            vmin, vmax = np.quantile(scores, [loq, hiq]) if len(scores) >= 10 else (scores.min(), scores.max())
            vmin, vmax = float(vmin), float(vmax if vmax > vmin else vmin + 1e-9)
            norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
            lc = LineCollection(
                segments,
                cmap=mpl.cm.get_cmap(cmap),
                norm=norm,
                linewidths=edge_lw,
                alpha=edge_alpha,
                zorder=1,
                rasterized=rasterize_edges,
            )
            lc.set_array(scores)
            ax.add_collection(lc)

            # Small, unobtrusive colorbar
            if color_bar:
                cbar = ax.figure.colorbar(lc, ax=ax, fraction=0.035, pad=0.02)
                cbar.set_label(score_key, rotation=90)
                cbar.outline.set_linewidth(0.5)
        else:
            lc = LineCollection(
                segments,
                colors=(0.2, .2, .2, .1),  # base color; alpha handled separately
                linewidths=edge_lw,
                alpha=edge_alpha,
                zorder=1,
                rasterized=rasterize_edges,
            )
            ax.add_collection(lc)

    # ---------------------------
    # 4) nodes (with white stroke for contrast)
    # ---------------------------
    xs = np.array([pos[n][0] for n in H.nodes if n in pos], dtype=float)
    ys = np.array([pos[n][1] for n in H.nodes if n in pos], dtype=float)

    ax.scatter(
        xs, ys,
        s=node_size,
        alpha=node_alpha,
         zorder=2,
         color=node_color,
        # linewidths=0.6,
        # edgecolors="white",
    )

    # ---------------------------
    # 5) cosmetics
    # ---------------------------
    ax.set_xlabel(time_key)
    # ax.set_ylabel("node index (within time)")
    ax.set_yticks([])

    if title is None:
        title = f"{score_key} ≥ {score_threshold:.2f}" + (f" (top {max_edges} edges)" if max_edges else "")
    ax.set_title(title, fontsize=11)

    # light grid helps readability without clutter
    ax.grid(True, axis="both", linewidth=0.5, alpha=0.25)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Creation time")

    if show_time_ticks:
        times = [t for t in buckets.keys() if t != float("inf")]
        if len(times) > 0:
            ax.set_xticks(sorted(set(times)))

    return H, pos

In [ ]:
TOTAL_SEEDS = 5 # adjust if you have fewer/more seeds per experiment

fig, axs = plt.subplots(TOTAL_SEEDS, len(EXP_BASE_NAMES), figsize=(3*len(EXP_BASE_NAMES), 3*TOTAL_SEEDS))

for base_name in EXP_BASE_NAMES:
    print(f"Plotting time-aligned graphs: {base_name} ", end="")
    for idx, phyl in enumerate(phylogeny[base_name]):
        print("#", end="", flush=True)
        # Hand metrics
        G_hand = to_nx_digraph(phyl[MODEL], artifacts=artifacts[base_name][idx].all_artifacts)
        draw_time_aligned_graph(G_hand, axs[idx, EXP_BASE_NAMES.index(base_name)], score_threshold=0.9, score_key="score", time_key="creation_time", jitter_y=0.1, show_time_ticks=False)

    axs[0, EXP_BASE_NAMES.index(base_name)].set_title(f"{PLOT_NAMES.get(base_name, base_name)}")
    print()
plt.tight_layout()
plt.show()